# Bees production 

### Setup

In [ ]:
import numpy as np
import pandas as pd 
import seaborn as sns
import utils
import matplotlib.pyplot as plt
import importlib
importlib.reload(utils)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder

honey_production_df, apistox_df, inspections_on_weather_dict = utils.load_bees_datasets()
honey_production_df_copy = honey_production_df.copy()


## Part 1: Honey production 

In [ ]:
honey_production_df = honey_production_df_copy.copy()

#cleaning the dataset
honey_production_df.drop(columns=["Unnamed: 0"], inplace=True)
for state in honey_production_df.state.unique():
    state_production = honey_production_df[honey_production_df.state == state]
    #keeping states that have data for all the 27 years from 1995 to 2021
    if len(state_production) < 27:
        honey_production_df.drop(state_production.index, inplace=True)

n_states = honey_production_df.state.nunique()
utils.EDA(honey_production_df, head = n_states) #to show the first data for all the considered states

honey_production_df_copy = honey_production_df.copy()

### Production Analysis 

In [ ]:
utils.plot_all(honey_production_df, "state", "year", "yield_per_colony")

### Production Trend Prediction

In [ ]:
#setting up data
years = honey_production_df['year'].unique()
production_means = honey_production_df.groupby("year")["production"].mean()

#preparing the model
X_train, X_test, y_train, y_test = train_test_split(years.reshape(-1,1), 
                                                    production_means, 
                                                    test_size=0.25,
                                                    random_state=30)
model= LinearRegression()
model.fit(X = X_train, y = y_train)
print(model.score(X_test, y_test))

#setting up the data for predictions
future_years = np.arange(2022, 2032)
years = np.concatenate([years,future_years])
y_pred = model.predict(years.reshape(-1, 1))

#plotting the results
plt.figure(figsize=(9, 5))
plt.plot(years, y_pred, color="red", label="Production")
plt.scatter(X_train, y_train, color="grey", s=40, alpha=0.7, label="Train")
plt.scatter(X_test, y_test, color="green", s=60, alpha=0.8, label="Test")
plt.xlabel("Year", fontsize = 11)
plt.ylabel("Production", fontsize = 11)
plt.title("Production Trend Prediction")
plt.legend()
plt.grid(True, alpha=0.35)
plt.show()

### Price analysis

In [ ]:
figure = plt.figure(figsize=(16,5))
plot1 = figure.add_subplot(1,2,1)
plot2 = figure.add_subplot(1,2,2)
plot1.plot(honey_production_df.year.unique(), honey_production_df.groupby("year")["stocks"].mean(), label = "stocks hold")
plot1.plot(honey_production_df.year.unique(), honey_production_df.groupby("year")["production"].mean(), label = "production")
plot2.plot(honey_production_df.year.unique(), honey_production_df.groupby("year")["value_of_production"].mean(), label = "Global price ($)", color = "green")
plot1.legend()
plot2.legend()
plot1.grid(alpha = 0.3)
plot2.grid(alpha = 0.3)

### Price trend Prediction 

In [ ]:
#setting up data
years = honey_production_df['year'].unique()
production_means = honey_production_df.groupby("year")["value_of_production"].mean()

#preparing the model
X_train, X_test, y_train, y_test = train_test_split(years.reshape(-1,1), 
                                                    production_means, 
                                                    test_size=0.25,
                                                    random_state=30)
model= LinearRegression()
model.fit(X = X_train, y = y_train)
print(model.score(X_test, y_test))

#setting up the data for predictions
future_years = np.arange(2022, 2032)
years = np.concatenate([years,future_years])
y_pred = model.predict(years.reshape(-1, 1))

#plotting the results
plt.figure(figsize=(9, 5))
plt.plot(years, y_pred, color="red", label="Price")
plt.scatter(X_train, y_train, color="grey", s=40, alpha=0.7, label="Train")
plt.scatter(X_test, y_test, color="green", s=60, alpha=0.8, label="Test")
plt.xlabel("Year", fontsize = 11)
plt.ylabel("Production", fontsize = 11)
plt.title("Production Trend Prediction")
plt.legend()
plt.grid(True, alpha=0.35)
plt.show()

## Part 2: Apistox 

In [ ]:
utils.EDA(apistox_df, head = 7)

### Further setup

In [ ]:
#the following row alone can be of interest
#pesticide_usage_df = utils.load_pesticide_usage_datasets()

pesticide_usage_df = utils.apistox_support_setup(remove_kg=False)

apistox_complete_df = apistox_df.merge(pesticide_usage_df)
apistox_complete_df.drop(["name", "source", "SMILES", "CID","other_agrochemical", "year"], axis = 1, inplace=True)
apistox_complete_df.columns = apistox_complete_df.columns.map(str.upper)

apistox_complete_df_copy = apistox_complete_df.copy()

### Apistox analysis

In [ ]:
utils.EDA(apistox_complete_df)

### Apistox Classification (Variety of Pesticides)

In [ ]:
apistox_complete_df = apistox_complete_df_copy
first_target = apistox_complete_df.PPDB_LEVEL
second_target = apistox_complete_df.LABEL

results, _ = utils.random_forest(apistox_complete_df[["HERBICIDE", "FUNGICIDE", "INSECTICIDE", "TOXICITY_TYPE"]],
                    first_target,
                    n_estimators=750,
                    random_state=42)
results

In [ ]:
apistox_complete_df = apistox_complete_df_copy

apistox_complete_df
results_feature_indexed = results.set_index("Features")
# #correctly giving weight to the information we are interested in
apistox_complete_df = apistox_complete_df.groupby("STATE_NAME")[["INSECTICIDE","HERBICIDE", "FUNGICIDE"]].sum()

apistox_complete_df["ORDER"] = 0
for column in apistox_complete_df.columns[:3]:
                                                            #:TODO fallo con iloc
    apistox_complete_df["ORDER"] += apistox_complete_df[column] * float(results_feature_indexed.loc[column])

apistox_complete_df = apistox_complete_df.sort_values("ORDER", ascending=False).drop("ORDER", axis = 1)
utils.bar_all(apistox_complete_df,apistox_complete_df.columns,["INSECTICIDE","HERBICIDE", "FUNGICIDE"])

variety_order = apistox_complete_df.reset_index()["STATE_NAME"].rename("VARIETY_ORDER")

### Apistox Analysis (Quantity of Pesticides)

In [ ]:
apistox_complete_df = apistox_complete_df_copy
first_target = apistox_complete_df.PPDB_LEVEL
second_target = apistox_complete_df.LABEL

results_label, _ = utils.random_forest(apistox_complete_df[["HERBICIDE", "FUNGICIDE", "INSECTICIDE", "TOXICITY_TYPE", "MEAN_KG"]],
                    first_target,
                    n_estimators=750,
                    random_state=42)
print("Impact on label:\n", results_label)
print("...........")
results_level, _ = utils.random_forest(apistox_complete_df[["HERBICIDE", "FUNGICIDE", "INSECTICIDE", "TOXICITY_TYPE", "MEAN_KG"]],
                    second_target,
                    n_estimators=750,
                    random_state=42)
print("Impact on PPDB_LEVEL:\n", results_level)



In [ ]:
#riordina states per mean kg e aggiungi colonna di stima di posizione
apistox_complete_df = apistox_complete_df_copy
apistox_complete_df = apistox_complete_df.groupby("STATE_NAME")[["MEAN_KG"]].sum().sort_values("MEAN_KG", ascending=False).reset_index()["STATE_NAME"].rename("QUANTITY_ORDER")
apistox_complete_df = pd.concat([variety_order, apistox_complete_df ], axis = 1)

variety_order_idx = apistox_complete_df.index.to_list()
quantity_order_idx = []
for state in apistox_complete_df.VARIETY_ORDER:
    quantity_order_idx.append(apistox_complete_df[apistox_complete_df["QUANTITY_ORDER"] == state].index[0])

apistox_complete_df["POS_DIFF"] = np.abs(pd.Series(variety_order_idx) - pd.Series(quantity_order_idx))

conditions = [
    apistox_complete_df["POS_DIFF"] == 0,
    apistox_complete_df["POS_DIFF"] <= 3, 
    apistox_complete_df["POS_DIFF"] <= 5,
    apistox_complete_df["POS_DIFF"] <= 10,
]

distance_labels = ["SAME POSITION", "VERY CLOSE", "CLOSE", "DISTANT"]

apistox_complete_df["POS_CHANGE"] = np.select(conditions, distance_labels, "VERY DISTANT")
apistox_complete_df.drop("POS_DIFF",axis=1, inplace =True)
apistox_complete_df

## Part 3: Weather Effects on Bees' Health


### Further Setup

#### Note
"HCC_Inspections": contains the inspections result, together with with the percentage of requirements met. An hive is health iff all requirements are set to 1.

    - Brood: all phases of a brood are present (egg, larvae, pupae)
    - Bees: there are enough bees in the hive to manage it and defend it
    - Queen: the queen is alive, young and can reproduce
    - Food: there's enough food and source of found outside
    - Stressors: there are no stressors 
    - Space: the space is safe, clean and not in detriment 

"Hourly_Weather": contains an analysis of the weather during the recorded hour.
Important notes:

    - Degrees are in Fahrenheit
    - Wind_Gust represents the peak of speed during a recorded interval

In [ ]:
# The dictionary containing the data for the third part of the analysis is loaded twice only for visualization purpose: 
# it is made possible to just use and visualize the dictionary with all the tables yet to be merged, if necessary.
table = inspections_on_weather_dict["HCC_Inspections"]
utils.EDA(table)

In [ ]:
inspections_on_weather_df = utils.load_complete_inspections_on_weather_df()

inspections_on_weather_df
inspections_on_weather_df_copy = inspections_on_weather_df.copy()

### Apiatory part analysis

In [ ]:
utils.EDA(inspections_on_weather_df)

In [ ]:
inspections_on_weather_df = inspections_on_weather_df_copy
failed_rows = inspections_on_weather_df[inspections_on_weather_df.HEALTHY == 0].shape[0]
failed_params = inspections_on_weather_df[inspections_on_weather_df.HEALTHY == 0][["BROOD", "BEES", "QUEEN", "FOOD", "STRESSORS", "SPACE"]].sum()
failed_params = failed_rows - failed_params 
plot = failed_params.plot(kind = "bar", 
                   title = f"Params failure during inspections (over {inspections_on_weather_df[inspections_on_weather_df.HEALTHY == 0].shape[0]})")
plot.tick_params(axis = "x", rotation = 35)

In [ ]:
inspections_on_weather_df = inspections_on_weather_df_copy
inspections_on_weather_df.groupby("STATE")[[ "BROOD", "BEES", "QUEEN", "FOOD", "STRESSORS", "SPACE", "HEALTHY"]].mean()

### Weather elements impact Classification

In [ ]:
inspections_on_weather_df = inspections_on_weather_df_copy
inspections_on_weather_df = inspections_on_weather_df.drop(["STATIONID", "WEATHERID", "OBSID", "APIARYID", "HIVEID", "INSPECTIONID"],axis = 1)
inspections_on_weather_df = inspections_on_weather_df.groupby(["STATE", "YEAR", "MONTH"])

inspections_on_weather_df = inspections_on_weather_df.mean(numeric_only=True).round(3)

plt.figure(figsize=(29,29))
sns.heatmap(inspections_on_weather_df.corr(), annot=True, cmap="coolwarm")
plt.show()

Even though NC uses more pesticides (both in variety and in quantity) than UT, and considering also that hives in Utah more often pass the inspections compared to the NC hives, the fact that NC's honey production is higher is more likely due to the weather and environmental condition in the territory: UT has less precipitation and the territory is basically dry and characterized by deserts, nulike NC who also faces the ocean and is mostly composed of plains than UT.

In [ ]:
inspections_on_weather_df = inspections_on_weather_df_copy
inspections_on_weather_df.groupby(["STATE", "YEAR"])[["PERCENT_MET", "TEMPERATURE", "HUMIDITY", "DEW_POINT", "WIND_SPEED", "WIND_GUST", "PRESSURE", "PRECIP"]].mean()